In [1]:
import pyroomacoustics as pra

import os
from tqdm import tqdm

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.autograd import profiler
import torchaudio
from torchmetrics.audio import SpeechReverberationModulationEnergyRatio, ShortTimeObjectiveIntelligibility


from einops import rearrange

from src.dataset import SignalDataset, TRUNetDataset
from src.loss import loss_tot, loss_MR, loss_MR_w
from models.fspen import FullSubPathExtension 

from IPython.display import Audio

from src.utils import model_eval, model_eval_fspen2x_ver3

import matplotlib.pyplot as plt

In [2]:
CHKP_DIR = "checkpoints"

np.set_printoptions(precision=3)
torch.set_printoptions(precision=3)

In [3]:
N_FFTS = 512
HOP_LENGTH = 256 # int(0.01625 * 16_000) # 256
# N_FFTS = 1024
# HOP_LENGTH = 512
SR = 16_000

DEVICE = "cpu" # torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"It's {DEVICE} time!!!")

It's cpu time!!!


In [5]:
from src.fspen_configs import TrainConfig48kHzEnc2x_ver2, TrainConfig48kHzEnc2x_enc_ext, TrainConfig

configs = TrainConfig()
# print(sum(configs.bands_num_in_groups), configs.dual_path_extension["num_modules"])
fspen = FullSubPathExtension(configs=configs)# .to(DEVICE)

state_d = torch.load(os.path.join(CHKP_DIR, "fspen_chkp", "TrainConfig_OG#2.pt"), map_location="cpu",  weights_only=False)

In [6]:
fspen.load_state_dict(state_d["model_state_dict"])

<All keys matched successfully>

In [7]:
state_d["plots"]["val loss"]

[0.005217811307654931,
 0.004068111956047897,
 0.003701605323630457,
 0.0036974331149115013,
 0.0034239371486294726,
 0.0031794396569379247,
 0.003196030819358734,
 0.0030953499685543086,
 0.0030734669864894105,
 0.0029887462643763195,
 0.002819613141652483,
 0.0027943036089149807,
 0.0029324633069336414,
 0.0028401228258959376,
 0.002875447918016177,
 0.002751991976625644,
 0.00275580865295174,
 0.0026418465243365904,
 0.0025806969766003583,
 0.002588699124037073,
 0.0025751419777337173,
 0.0025392585039998475,
 0.002581683891968658,
 0.0026425469057777752,
 0.00262738865477821,
 0.0025792015232862188,
 0.00253463051138589,
 0.0025471642774601397,
 0.0025146752058600006,
 0.0024656496780852857,
 0.002499896294186608,
 0.0024425830208481504,
 0.0024486621394037055,
 0.002586523285852029,
 0.0024570810626475858,
 0.0024724137440968593,
 0.002407626625007162,
 0.0024373104634623113,
 0.0023928471894648215,
 0.0024456112923172233,
 0.0024270866997539997,
 0.002462234491339097,
 0.00232790

In [8]:
def vorbis_window(winlen, device="cuda"):
    sq = torch.sin(torch.pi/2*(torch.sin(torch.pi/winlen*(torch.arange(winlen)-0.5))**2)).float()
    return sq

In [9]:
AUDIO_PATH = "input_sig_part.wav"

In [9]:
signal, signal_sr = torchaudio.load(AUDIO_PATH)

input_signal, _ = SignalDataset.normalize_audio(signal)

In [10]:
window = vorbis_window(N_FFTS)

spec = torch.stft(
            input_signal,
            n_fft=N_FFTS,
            hop_length=HOP_LENGTH,
            # onesided=True,
            win_length=N_FFTS,
            window=window,
            return_complex=True,
            normalized=True,
            center=True
        )

output, _ = model_eval(fspen, spec, configs, DEVICE, hid_size=64)

out_wave = torch.istft(output, n_fft=N_FFTS, hop_length=HOP_LENGTH, win_length=N_FFTS,
                       window=window,
                       # onesided=True,
                       return_complex=False,
                       normalized=True,
                       center=True)

out_wave = out_wave.reshape(-1)

In [11]:
from scipy.io.wavfile import write

write(AUDIO_PATH[:-4] + "_overfit.wav", SR, out_wave.cpu().detach().numpy())

In [12]:
import yaml

from NISQA_s.src.core.model_torch import model_init
from NISQA_s.src.utils.process_utils import process

NISQA_PATH = "NISQA_s/config/nisqa_s.yaml"

with open(NISQA_PATH, 'r') as stream:
    nisqa_args = yaml.safe_load(stream)
nisqa_args["ms_n_fft"] = 512
nisqa_args["hop_length"] = 256
nisqa_args["ms_win_length"] = 512
nisqa_args["ckp"] = nisqa_args["ckp"][3:]

nisqa, h0_nisqa, c0_nisqa = model_init(nisqa_args)

/home/zakhar/miniconda3/envs/ems_dereverb/lib/python3.10/site-packages/torch/nn/modules/rnn.py:83: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=1 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


In [13]:
from torch_stoi import NegSTOILoss

srmr = SpeechReverberationModulationEnergyRatio(fs=16_000, norm=True)
stoi = NegSTOILoss(SR, use_vad=False, do_resample=False).to(DEVICE)

In [14]:
target, signal_sr = torchaudio.load("data/test_audio/p258_005.wav")

In [15]:
from torchaudio.transforms import Resample

min_l = min(out_wave.shape[-1], signal.shape[-1])
nisqa_score_in, _, _ = process(signal.unsqueeze(0), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)
nisqa_score_out, _, _ = process(out_wave.unsqueeze(0), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)
print("NISQA in:", nisqa_score_in)
print("NISQA out:", nisqa_score_out)

stoi_score_in = -stoi(signal[..., :min_l], target[..., :min_l])
stoi_score_out = -stoi(out_wave[..., :min_l].unsqueeze(0), target[..., :min_l])
print(f"STOI in:", stoi_score_in.item())
print(f"STOI out:", stoi_score_out.item())

resampler = Resample(SR, 16_000)
signal = resampler(signal.cpu())
output = resampler(out_wave.cpu())
# target = resampler(target.cpu()).cuda()
# min_l = min(output.shape[-1], target.shape[-1])

srmr_score_in = srmr(signal.detach())
srmr_score_out = srmr(output.detach())
print(f"SRMR in: {srmr_score_in:.2f}")
print(f"SRMR out: {srmr_score_out:.2f}")
# try:
#     pesq_score = pesq(output[..., :min_l], target[..., :min_l])

NISQA in: tensor([[1.662, 1.975, 2.909, 2.432, 2.451]])
NISQA out: tensor([[3.358, 2.249, 3.951, 3.771, 3.819]])
STOI in: 0.828109085559845
STOI out: 0.8199795484542847
SRMR in: 2.47
SRMR out: 2.38


In [16]:
out_wave - signal

RuntimeError: The size of tensor a (239616) must match the size of tensor b (79872) at non-singleton dimension 1